# A/B Testing Framework for ML Models

A simulation-based experimentation framework for comparing anomaly detection model variants.  
Computes statistical significance, effect sizes, and power analysis — with a Streamlit dashboard.

**Stack:** Python · SciPy · Statsmodels · Pandas · Matplotlib · Streamlit

---
## Use Cases
- Compare two anomaly detection threshold strategies
- Validate a new model against baseline before production rollout
- Run power analysis to determine required sample size
- Compute false positive rate tradeoffs between model variants

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest
from sklearn.metrics import precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
print("Imports OK")

## 2. Simulate Model Outputs — Control vs Treatment

In [ ]:
def simulate_model_outputs(n=1000, true_anomaly_rate=0.08,
                           control_precision=0.72, control_recall=0.80,
                           treat_precision=0.83,   treat_recall=0.85,
                           seed=42):
    """
    Simulate binary anomaly detection outputs for two model variants.
    Returns ground truth and predictions for Control (A) and Treatment (B).
    """
    np.random.seed(seed)
    ground_truth = (np.random.rand(n) < true_anomaly_rate).astype(int)
    n_actual = ground_truth.sum()

    def make_preds(precision, recall, gt):
        preds = np.zeros(len(gt), dtype=int)
        pos_idx = np.where(gt == 1)[0]
        neg_idx = np.where(gt == 0)[0]
        # True positives
        tp = int(len(pos_idx) * recall)
        preds[np.random.choice(pos_idx, tp, replace=False)] = 1
        # False positives
        fp = int(tp / precision - tp) if precision > 0 else 0
        fp = min(fp, len(neg_idx))
        preds[np.random.choice(neg_idx, fp, replace=False)] = 1
        return preds

    preds_A = make_preds(control_precision, control_recall, ground_truth)
    preds_B = make_preds(treat_precision,   treat_recall,   ground_truth)
    return ground_truth, preds_A, preds_B

gt, preds_A, preds_B = simulate_model_outputs()
print(f"Ground truth positives : {gt.sum()}")
print(f"Control (A) detections : {preds_A.sum()}")
print(f"Treatment (B) detections: {preds_B.sum()}")

## 3. Metric Computation

In [ ]:
def compute_metrics(gt, preds, name):
    p  = precision_score(gt, preds, zero_division=0)
    r  = recall_score(gt, preds, zero_division=0)
    f1 = f1_score(gt, preds, zero_division=0)
    fp_rate = ((preds == 1) & (gt == 0)).sum() / (gt == 0).sum()
    fn_rate = ((preds == 0) & (gt == 1)).sum() / (gt == 1).sum()
    return {"Model": name, "Precision": p, "Recall": r,
            "F1": f1, "FP Rate": fp_rate, "FN Rate": fn_rate}

metrics_A = compute_metrics(gt, preds_A, "Control (A)")
metrics_B = compute_metrics(gt, preds_B, "Treatment (B)")
metrics_df = pd.DataFrame([metrics_A, metrics_B]).set_index("Model")
print(metrics_df.round(4))

## 4. Statistical Significance Tests

In [ ]:
def run_significance_tests(gt, preds_A, preds_B, alpha=0.05):
    results = {}

    # --- 1. Proportions Z-Test on Precision ---
    tp_A = ((preds_A == 1) & (gt == 1)).sum()
    tp_B = ((preds_B == 1) & (gt == 1)).sum()
    det_A = preds_A.sum()
    det_B = preds_B.sum()

    if det_A > 0 and det_B > 0:
        z_prec, p_prec = proportions_ztest([tp_A, tp_B], [det_A, det_B])
        results["Precision Z-test"] = {
            "statistic": round(z_prec, 4), "p_value": round(p_prec, 4),
            "significant": p_prec < alpha
        }

    # --- 2. Proportions Z-Test on Recall (TPR) ---
    n_pos = gt.sum()
    z_rec, p_rec = proportions_ztest([tp_A, tp_B], [n_pos, n_pos])
    results["Recall Z-test"] = {
        "statistic": round(z_rec, 4), "p_value": round(p_rec, 4),
        "significant": p_rec < alpha
    }

    # --- 3. McNemar's Test (paired comparison on same data) ---
    b = ((preds_A == 1) & (preds_B == 0)).sum()  # A correct, B wrong
    c = ((preds_A == 0) & (preds_B == 1)).sum()  # B correct, A wrong
    if b + c > 0:
        chi2 = (abs(b - c) - 1) ** 2 / (b + c)
        p_mc = 1 - stats.chi2.cdf(chi2, df=1)
        results["McNemar Test"] = {
            "statistic": round(chi2, 4), "p_value": round(p_mc, 4),
            "significant": p_mc < alpha
        }

    return pd.DataFrame(results).T

sig_df = run_significance_tests(gt, preds_A, preds_B)
print("Statistical Significance Tests (alpha=0.05)")
print(sig_df)

## 5. Effect Size & Power Analysis

In [ ]:
def cohens_h(p1, p2):
    """Effect size for proportions."""
    return 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))

prec_A = metrics_A["Precision"]
prec_B = metrics_B["Precision"]
h      = cohens_h(prec_B, prec_A)
print(f"Cohen's h (precision uplift): {h:.4f}")
if abs(h) < 0.2:   print("Effect size: Small")
elif abs(h) < 0.5: print("Effect size: Medium")
else:              print("Effect size: Large")

# Sample size needed to detect this effect
power_analysis = TTestIndPower()
required_n = power_analysis.solve_power(effect_size=abs(h), alpha=0.05, power=0.80)
print(f"\nSample size needed per group (power=0.80, alpha=0.05): {int(np.ceil(required_n))}")

# Power curve
sample_sizes = np.arange(50, 2000, 50)
powers = [power_analysis.solve_power(effect_size=abs(h), alpha=0.05, nobs1=n) for n in sample_sizes]

plt.figure(figsize=(10, 4))
plt.plot(sample_sizes, powers, linewidth=2)
plt.axhline(0.80, color="red", linestyle="--", label="Power = 0.80")
plt.axvline(required_n, color="green", linestyle="--", label=f"Required n = {int(required_n)}")
plt.xlabel("Sample Size per Group")
plt.ylabel("Statistical Power")
plt.title("Power Curve — Precision Uplift Test")
plt.legend()
plt.tight_layout()
plt.savefig("power_curve.png", dpi=150)
plt.show()

## 6. Threshold Comparison Sweep

In [ ]:
def threshold_sweep(scores_A, scores_B, gt, thresholds=np.linspace(0.1, 0.9, 50)):
    """Compare F1 across thresholds for two score distributions."""
    f1_A, f1_B = [], []
    for t in thresholds:
        f1_A.append(f1_score(gt, (scores_A > t).astype(int), zero_division=0))
        f1_B.append(f1_score(gt, (scores_B > t).astype(int), zero_division=0))
    return pd.DataFrame({"threshold": thresholds, "F1_Control": f1_A, "F1_Treatment": f1_B})

# Simulate continuous anomaly scores
scores_A = np.where(gt == 1, np.random.beta(5, 2, len(gt)), np.random.beta(2, 5, len(gt)))
scores_B = np.where(gt == 1, np.random.beta(6, 2, len(gt)), np.random.beta(2, 6, len(gt)))

sweep_df = threshold_sweep(scores_A, scores_B, gt)

plt.figure(figsize=(10, 4))
plt.plot(sweep_df["threshold"], sweep_df["F1_Control"],   label="Control (A)", linewidth=2)
plt.plot(sweep_df["threshold"], sweep_df["F1_Treatment"], label="Treatment (B)", linewidth=2, linestyle="--")
plt.xlabel("Threshold")
plt.ylabel("F1 Score")
plt.title("F1 Score vs Threshold — Control vs Treatment")
plt.legend()
plt.tight_layout()
plt.savefig("threshold_sweep.png", dpi=150)
plt.show()

best_A = sweep_df.loc[sweep_df["F1_Control"].idxmax()]
best_B = sweep_df.loc[sweep_df["F1_Treatment"].idxmax()]
print(f"Best threshold A: {best_A['threshold']:.2f}  F1={best_A['F1_Control']:.4f}")
print(f"Best threshold B: {best_B['threshold']:.2f}  F1={best_B['F1_Treatment']:.4f}")

## 7. Experiment Summary Report

In [ ]:
print("="*55)
print("       A/B EXPERIMENT SUMMARY REPORT")
print("="*55)
print(f"\n{'Metric':<20} {'Control (A)':>15} {'Treatment (B)':>15} {'Delta':>10}")
print("-"*55)
for col in ["Precision", "Recall", "F1", "FP Rate"]:
    a = metrics_df.loc["Control (A)", col]
    b = metrics_df.loc["Treatment (B)", col]
    delta = b - a
    print(f"{col:<20} {a:>15.4f} {b:>15.4f} {delta:>+10.4f}")
print("\nStatistical Significance:")
print(sig_df[["p_value", "significant"]].to_string())
print(f"\nEffect size (Cohen's h): {h:.4f}")
print(f"Min sample size needed : {int(np.ceil(required_n))} per group")
print("="*55)

## 8. Streamlit Dashboard

Run the dashboard with:
```bash
streamlit run dashboard.py
```
See `dashboard.py` in this repo for the interactive UI.